In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
import xgboost as xgb
from xgboost import XGBClassifier
from sklearn.metrics import confusion_matrix,roc_auc_score,roc_curve,classification_report
from sklearn.preprocessing import LabelEncoder

In [2]:
data=pd.read_csv("/content/digital_marketing_campaign_dataset.csv")
df=pd.DataFrame(data)
print(df.head())
print(df.isnull().sum())
print(df.groupby("Conversion")["ConversionRate"].describe())
print(df.groupby("Conversion")["ClickThroughRate"].describe())

   CustomerID  Age  Gender  Income CampaignChannel CampaignType      AdSpend  \
0        8000   56  Female  136912    Social Media    Awareness  6497.870068   
1        8001   69    Male   41760           Email    Retention  3898.668606   
2        8002   46  Female   88456             PPC    Awareness  1546.429596   
3        8003   32  Female   44085             PPC   Conversion   539.525936   
4        8004   60  Female   83964             PPC   Conversion  1678.043573   

   ClickThroughRate  ConversionRate  WebsiteVisits  PagesPerVisit  TimeOnSite  \
0          0.043919        0.088031              0       2.399017    7.396803   
1          0.155725        0.182725             42       2.917138    5.352549   
2          0.277490        0.076423              2       8.223619   13.794901   
3          0.137611        0.088004             47       4.540939   14.688363   
4          0.252851        0.109940              0       2.046847   13.993370   

   SocialShares  EmailOpens  Ema

In [3]:
encoder=LabelEncoder()
df["Gender"]=encoder.fit_transform(df["Gender"])
df=pd.get_dummies(df,columns=["CampaignChannel","CampaignType"])
df["email_engagement"] = df["EmailClicks"] / (df["EmailOpens"] + 1)
df["engagement_score"] = (df["PagesPerVisit"]*df["TimeOnSite"])
df["customer_value"] = (df["PreviousPurchases"]*df["LoyaltyPoints"])
df["marketing_efficiency"] = (df["ClickThroughRate"]*df["TimeOnSite"])
df["spend_per_visit"] = (df["AdSpend"]/(df["WebsiteVisits"]+1))
X=df.drop(["CustomerID","Conversion","AdvertisingPlatform","AdvertisingTool"],axis=1)
Y=df["Conversion"]
X_train,X_test,Y_train,Y_test=train_test_split(X,Y,test_size=0.2,random_state=42)

In [4]:

clf = XGBClassifier()
clf.fit(X_train,Y_train)
y_pred = clf.predict(X_test)
print(confusion_matrix(Y_test,y_pred))
print(classification_report(Y_test,y_pred))
y_prob = clf.predict_proba(X_test)[:,1]

print(roc_auc_score(Y_test, y_prob))

[[  89  105]
 [  32 1374]]
              precision    recall  f1-score   support

           0       0.74      0.46      0.57       194
           1       0.93      0.98      0.95      1406

    accuracy                           0.91      1600
   macro avg       0.83      0.72      0.76      1600
weighted avg       0.91      0.91      0.91      1600

0.8120353125779062


In [8]:
importance = pd.DataFrame({
    "feature": X.columns,
    "importance": clf.feature_importances_
}).sort_values(
    by="importance",
    ascending=False
)

print(importance.head(20))

                         feature  importance
21       CampaignType_Conversion    0.069059
11                   EmailClicks    0.065715
10                    EmailOpens    0.055037
25                customer_value    0.047097
26          marketing_efficiency    0.046314
12             PreviousPurchases    0.046298
3                        AdSpend    0.044490
6                  WebsiteVisits    0.044331
5                 ConversionRate    0.043383
7                  PagesPerVisit    0.043234
4               ClickThroughRate    0.039340
24              engagement_score    0.038629
18  CampaignChannel_Social Media    0.035445
8                     TimeOnSite    0.032954
13                 LoyaltyPoints    0.032205
14         CampaignChannel_Email    0.029936
15           CampaignChannel_PPC    0.029007
9                   SocialShares    0.026975
1                         Gender    0.025803
27               spend_per_visit    0.025708
